# 06 - SHAP Explainability for Frozen CardioTwin Models

This notebook builds judge-ready and API-ready SHAP explanations for frozen CardioTwin ML version `1.0.0`. It does not retrain, tune, change features, change calibration, or change decision thresholds.

SHAP values describe how features contributed to the model prediction. They do not establish medical causality.

In [1]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

started = time.perf_counter()
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "ml" else CWD
if not (PROJECT_ROOT / "ml").exists():
    PROJECT_ROOT = next(path for path in [CWD, *CWD.parents] if (path / "ml").exists() and (path / "docs").exists())
if str(PROJECT_ROOT / "ml") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "ml"))

from src.explainability.shap_explainer import CardioTwinExplainer, TARGETS, positive_probability

ARTIFACT_DIR = PROJECT_ROOT / "ml" / "artifacts" / "metrics" / "explainability"
FIGURE_DIR = ARTIFACT_DIR / "figures"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)
print(PROJECT_ROOT)

C:\Users\samri\cod\git\IIT_Mandi\cardio-3d-ai


## Load Frozen Models and Metadata

The explainer reloads the saved calibrated joblib pipelines, `model_metadata.json`, and `input_schema.json`. It rejects any model version other than `1.0.0`.

In [2]:
explainer = CardioTwinExplainer(project_root=PROJECT_ROOT, background_size=80, random_state=2026)
assert explainer.metadata["model_version"] == "1.0.0"
for target in TARGETS:
    model = explainer.models[target]
    probs = positive_probability(model, explainer.X.head(5))
    assert np.isfinite(probs).all()
    assert ((probs >= 0) & (probs <= 1)).all()
    print(target.upper(), type(model).__name__, probs.round(4).tolist())

CAD CalibratedClassifierCV [0.7362, 0.991, 0.9448, 0.3564, 0.2312]


LAD CalibratedClassifierCV [0.4483, 0.9356, 0.9083, 0.3476, 0.1944]
LCX CalibratedClassifierCV [0.2205, 0.6988, 0.3395, 0.1925, 0.143]


RCA CalibratedClassifierCV [0.3403, 0.3446, 0.3266, 0.2108, 0.1308]


## Pipeline Inspection

The saved artifacts are `CalibratedClassifierCV` objects. SHAP is applied to the fitted underlying estimators inside the calibration wrapper, after frozen preprocessing and any target-specific feature selection. Calibrated probabilities still come from the saved deployment pipeline.

In [3]:
inspection_rows = []
for target in TARGETS:
    meta = explainer.metadata["models"][target]
    for context in explainer.contexts[target]:
        inspection_rows.append({
            "target": target,
            "estimator_index": context.estimator_index,
            "algorithm": meta["algorithm"],
            "explainer": context.explainer_name,
            "excluded_features": ", ".join(meta["excluded_features"]) or "none",
            "feature_selection": meta["feature_selection"],
            "n_transformed_features": len(context.transformed_feature_names),
        })
inspection = pd.DataFrame(inspection_rows)
inspection.to_csv(ARTIFACT_DIR / "pipeline_inspection.csv", index=False)
inspection

,target,estimator_index,algorithm,explainer,excluded_features,feature_selection,n_transformed_features
0,cad,0,LogisticRegression,shap.LinearExplainer,Lymph,none,93
1,cad,1,LogisticRegression,shap.LinearExplainer,Lymph,none,95
2,cad,2,LogisticRegression,shap.LinearExplainer,Lymph,none,94
3,cad,3,LogisticRegression,shap.LinearExplainer,Lymph,none,95
4,cad,4,LogisticRegression,shap.LinearExplainer,Lymph,none,95
5,lad,0,RandomForestClassifier,shap.TreeExplainer,none,none,94
6,lad,1,RandomForestClassifier,shap.TreeExplainer,none,none,96
7,lad,2,RandomForestClassifier,shap.TreeExplainer,none,none,96
8,lad,3,RandomForestClassifier,shap.TreeExplainer,none,none,95
9,lad,4,RandomForestClassifier,shap.TreeExplainer,none,none,96


## Global SHAP Importance

Both transformed-feature and clinical-feature aggregated importance are exported. Dashboard views should prefer the aggregated clinical-feature table.

In [4]:
global_importance, transformed_importance = explainer.all_global_importance()
global_importance.to_csv(ARTIFACT_DIR / "global_shap_importance.csv", index=False)
transformed_importance.to_csv(ARTIFACT_DIR / "global_shap_transformed_importance.csv", index=False)

for target in TARGETS:
    explainer.plot_global_importance(target, FIGURE_DIR / f"{target}_global_shap.png", top_n=15)
    explainer.plot_beeswarm(target, FIGURE_DIR / f"{target}_beeswarm.png", max_display=15)

for target in TARGETS:
    print(f"\n{target.upper()} top 10 aggregated clinical features")
    display(global_importance[global_importance["target"] == target].head(10))


CAD top 10 aggregated clinical features


,target,feature,mean_abs_shap,rank
0,cad,Typical Chest Pain,0.659398,1
1,cad,Age,0.463666,2
2,cad,TG,0.271876,3
3,cad,HTN,0.252789,4
4,cad,Region RWMA,0.250025,5
5,cad,Tinversion,0.243503,6
6,cad,DM,0.234686,7
7,cad,Atypical,0.234603,8
8,cad,Dyspnea,0.229536,9
9,cad,EF-TTE,0.211854,10



LAD top 10 aggregated clinical features


,target,feature,mean_abs_shap,rank
54,lad,Typical Chest Pain,0.106800,1
55,lad,Region RWMA,0.041538,2
56,lad,Atypical,0.033169,3
57,lad,Age,0.022411,4
58,lad,EF-TTE,0.019608,5
59,lad,Tinversion,0.015591,6
60,lad,Neut,0.012440,7
61,lad,Lymph,0.011836,8
62,lad,ESR,0.009978,9
63,lad,TG,0.009734,10



LCX top 10 aggregated clinical features


,target,feature,mean_abs_shap,rank
109,lcx,Age,0.416326,1
110,lcx,Typical Chest Pain,0.391688,2
111,lcx,Na,0.163603,3
112,lcx,BP,0.145883,4
113,lcx,TG,0.144972,5
114,lcx,DM,0.135957,6
115,lcx,PLT,0.124297,7
116,lcx,CR,0.121332,8
117,lcx,Sex,0.108503,9
118,lcx,Length,0.092992,10



RCA top 10 aggregated clinical features


,target,feature,mean_abs_shap,rank
159,rca,Age,0.338931,1
160,rca,DM,0.319314,2
161,rca,Typical Chest Pain,0.268759,3
162,rca,Sex,0.247889,4
163,rca,ESR,0.233814,5
164,rca,TG,0.186348,6
165,rca,Dyspnea,0.181318,7
166,rca,Neut,0.168750,8
167,rca,BUN,0.144721,9
168,rca,Function Class,0.135492,10


## Representative Example Patients

Rows below are selected only to demonstrate local explanations and plotting. They are not additional validation and should not be interpreted as a performance claim.

In [5]:
def prediction_frame(target: str) -> pd.DataFrame:
    y = explainer.y[target].astype(int).to_numpy()
    p = positive_probability(explainer.models[target], explainer.X)
    threshold = explainer.threshold(target)
    pred = (p >= threshold).astype(int)
    return pd.DataFrame({
        "patient_index": np.arange(len(explainer.X)),
        "target": target,
        "y_true": y,
        "probability": p,
        "prediction": pred,
        "threshold": threshold,
    })

predictions = pd.concat([prediction_frame(target) for target in TARGETS], ignore_index=True)
predictions.to_csv(ARTIFACT_DIR / "example_candidate_predictions.csv", index=False)

def first_correct(target: str, label: int) -> int:
    subset = predictions[(predictions.target == target) & (predictions.y_true == label) & (predictions.prediction == label)]
    if subset.empty:
        subset = predictions[(predictions.target == target) & (predictions.y_true == label)]
    if subset.empty:
        subset = predictions[predictions.target == target]
    return int(subset.sort_values("probability", ascending=(label == 0)).iloc[0]["patient_index"])

example_indices = {
    "cad_positive": first_correct("cad", 1),
    "cad_negative": first_correct("cad", 0),
    "lad_positive": first_correct("lad", 1),
    "lcx_positive": first_correct("lcx", 1),
    "rca_positive": first_correct("rca", 1),
}
example_indices

{'cad_positive': 174,
 'cad_negative': 8,
 'lad_positive': 240,
 'lcx_positive': 87,
 'rca_positive': 114}

## Patient-Level Explanations and Waterfall Plots

In [6]:
def slim_explanation(name: str, patient_index: int, target: str) -> dict:
    patient = explainer.X.iloc[[patient_index]]
    explanation = explainer.explain_target(patient, target, top_n=5)
    return {
        "example_name": name,
        "patient_index": int(patient_index),
        "target": explanation["target"],
        "calibrated_probability": explanation["probability"],
        "threshold": explanation["threshold"],
        "classification": explanation["classification"],
        "top_increasing_contributors": explanation["top_risk_factors"],
        "top_decreasing_contributors": explanation["top_decreasing_risk_factors"],
        "base_value": explanation["base_value"],
        "explanation_method": explanation["explanation_method"],
        "explanation_scope": explanation["explanation_scope"],
    }

example_specs = [
    ("cad_positive", example_indices["cad_positive"], "cad", "cad_patient_positive_waterfall.png"),
    ("cad_negative", example_indices["cad_negative"], "cad", "cad_patient_negative_waterfall.png"),
    ("lad_positive", example_indices["lad_positive"], "lad", "lad_patient_waterfall.png"),
    ("lcx_positive", example_indices["lcx_positive"], "lcx", "lcx_patient_waterfall.png"),
    ("rca_positive", example_indices["rca_positive"], "rca", "rca_patient_waterfall.png"),
]

example_explanations = []
for name, patient_index, target, figure_name in example_specs:
    example_explanations.append(slim_explanation(name, patient_index, target))
    explainer.plot_waterfall(explainer.X.iloc[[patient_index]], target, FIGURE_DIR / figure_name, max_display=12)

with open(ARTIFACT_DIR / "example_patient_explanations.json", "w", encoding="utf-8") as f:
    json.dump(example_explanations, f, indent=2)

print(json.dumps(example_explanations[0], indent=2)[:2500])

{
  "example_name": "cad_positive",
  "patient_index": 174,
  "target": "CAD",
  "calibrated_probability": 0.9984376006651331,
  "threshold": 0.3,
  "classification": "positive",
  "top_increasing_contributors": [
    {
      "feature": "EF-TTE",
      "raw_value": 20,
      "contribution": 0.8314802960069696,
      "direction": "increases_model_score",
      "ui_label": "Increases predicted risk"
    },
    {
      "feature": "Typical Chest Pain",
      "raw_value": 1,
      "contribution": 0.629022918261511,
      "direction": "increases_model_score",
      "ui_label": "Increases predicted risk"
    },
    {
      "feature": "DM",
      "raw_value": 1,
      "contribution": 0.4163509232304726,
      "direction": "increases_model_score",
      "ui_label": "Increases predicted risk"
    },
    {
      "feature": "Tinversion",
      "raw_value": 1,
      "contribution": 0.3831691185498897,
      "direction": "increases_model_score",
      "ui_label": "Increases predicted risk"
    },
  

## Consistency Checks

In [7]:
validation = explainer.validate()
with open(ARTIFACT_DIR / "explainability_validation.json", "w", encoding="utf-8") as f:
    json.dump(validation, f, indent=2)

assert validation["model_version"] == "1.0.0"
for target, checks in validation["targets"].items():
    assert checks["predict_proba_valid"]
    assert checks["excluded_features_absent"]
    assert checks["finite_shap"]
    if target == "lcx":
        assert checks["lcx_select_k_correct"]
    for context in checks["contexts"]:
        tolerance = 1e-6 if context["explainer"] == "shap.LinearExplainer" else 1e-3
        assert context["additive_max_abs_error"] < tolerance
validation

{'model_version': '1.0.0',
 'targets': {'cad': {'predict_proba_valid': True,
   'contexts': [{'estimator_index': 0,
     'explainer': 'shap.LinearExplainer',
     'n_transformed_features': 93,
     'shap_shape': [3, 93],
     'finite_shap': True,
     'excluded_features_absent': True,
     'additive_max_abs_error': 3.7470027081099033e-16},
    {'estimator_index': 1,
     'explainer': 'shap.LinearExplainer',
     'n_transformed_features': 95,
     'shap_shape': [3, 95],
     'finite_shap': True,
     'excluded_features_absent': True,
     'additive_max_abs_error': 1.7763568394002505e-15},
    {'estimator_index': 2,
     'explainer': 'shap.LinearExplainer',
     'n_transformed_features': 94,
     'shap_shape': [3, 94],
     'finite_shap': True,
     'excluded_features_absent': True,
     'additive_max_abs_error': 2.7755575615628914e-16},
    {'estimator_index': 3,
     'explainer': 'shap.LinearExplainer',
     'n_transformed_features': 95,
     'shap_shape': [3, 95],
     'finite_shap': 

## API-Ready JSON and Latency

`explain_all()` returns JSON-serializable explanations for the same patient across all targets. This is the intended FastAPI-facing interface.

In [8]:
sample_patient = explainer.X.iloc[[example_indices["cad_positive"]]]
all_explanation = explainer.explain_all(sample_patient, top_n=3)
json.dumps(all_explanation)
with open(ARTIFACT_DIR / "example_explain_all.json", "w", encoding="utf-8") as f:
    json.dump(all_explanation, f, indent=2)

latency = explainer.latency_report(sample_patient, repeats=5, top_n=5)
latency.to_csv(ARTIFACT_DIR / "explanation_latency.csv", index=False)
print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk != "transformed_contributions"} for k, v in all_explanation.items()}, indent=2)[:3500])
latency

{
  "CAD": {
    "target": "CAD",
    "probability": 0.9984376006651331,
    "threshold": 0.3,
    "classification": "positive",
    "top_risk_factors": [
      {
        "feature": "EF-TTE",
        "raw_value": 20,
        "contribution": 0.8314802960069696,
        "direction": "increases_model_score",
        "ui_label": "Increases predicted risk"
      },
      {
        "feature": "Typical Chest Pain",
        "raw_value": 1,
        "contribution": 0.629022918261511,
        "direction": "increases_model_score",
        "ui_label": "Increases predicted risk"
      },
      {
        "feature": "DM",
        "raw_value": 1,
        "contribution": 0.4163509232304726,
        "direction": "increases_model_score",
        "ui_label": "Increases predicted risk"
      }
    ],
    "top_protective_factors": [
      {
        "feature": "FH",
        "raw_value": 0,
        "contribution": -0.08921486764946107,
        "direction": "decreases_model_score",
        "ui_label": "Decrease

,target,mean_seconds,min_seconds,max_seconds,repeats
0,cad,0.467390,0.327562,0.836543,5
1,lad,0.594668,0.515643,0.649876,5
2,lcx,0.443709,0.366790,0.606361,5
3,rca,0.480465,0.349618,0.672411,5


## Final Summary

In [9]:
figure_paths = sorted(str(path.relative_to(PROJECT_ROOT)) for path in FIGURE_DIR.glob("*.png"))
artifact_paths = sorted(str(path.relative_to(PROJECT_ROOT)) for path in ARTIFACT_DIR.glob("*") if path.is_file())
explainer_used = {target.upper(): explainer.contexts[target][0].explainer_name for target in TARGETS}
execution_seconds = time.perf_counter() - started

summary = {
    "cad_top_10_global_features": global_importance[global_importance.target == "cad"].head(10)[["feature", "mean_abs_shap"]].to_dict("records"),
    "lad_top_10_global_features": global_importance[global_importance.target == "lad"].head(10)[["feature", "mean_abs_shap"]].to_dict("records"),
    "lcx_top_10_global_features": global_importance[global_importance.target == "lcx"].head(10)[["feature", "mean_abs_shap"]].to_dict("records"),
    "rca_top_10_global_features": global_importance[global_importance.target == "rca"].head(10)[["feature", "mean_abs_shap"]].to_dict("records"),
    "shap_explainer_used": explainer_used,
    "lcx_selectkbest_mapping_correct": validation["targets"]["lcx"]["lcx_select_k_correct"],
    "target_specific_exclusions_respected": {target.upper(): validation["targets"][target]["excluded_features_absent"] for target in TARGETS},
    "example_explain_all_json_keys": list(all_explanation.keys()),
    "explanation_latency_seconds": latency.to_dict("records"),
    "generated_artifact_paths": artifact_paths,
    "generated_figure_paths": figure_paths,
    "warnings_errors": [],
    "execution_seconds": execution_seconds,
}
with open(ARTIFACT_DIR / "shap_run_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2)[:12000])

{
  "cad_top_10_global_features": [
    {
      "feature": "Typical Chest Pain",
      "mean_abs_shap": 0.6593978307639771
    },
    {
      "feature": "Age",
      "mean_abs_shap": 0.46366630823080773
    },
    {
      "feature": "TG",
      "mean_abs_shap": 0.27187575512007384
    },
    {
      "feature": "HTN",
      "mean_abs_shap": 0.25278911163017714
    },
    {
      "feature": "Region RWMA",
      "mean_abs_shap": 0.25002462627863087
    },
    {
      "feature": "Tinversion",
      "mean_abs_shap": 0.24350278050860236
    },
    {
      "feature": "DM",
      "mean_abs_shap": 0.23468602739962804
    },
    {
      "feature": "Atypical",
      "mean_abs_shap": 0.23460322015093493
    },
    {
      "feature": "Dyspnea",
      "mean_abs_shap": 0.22953589381756703
    },
    {
      "feature": "EF-TTE",
      "mean_abs_shap": 0.21185414027609722
    }
  ],
  "lad_top_10_global_features": [
    {
      "feature": "Typical Chest Pain",
      "mean_abs_shap": 0.10680001025082277